# LangGraph Simple Agent with Llama-3.2-1B-Instruct

This notebook demonstrates a simple conversational agent built with **LangGraph** and **Llama-3.2-1B-Instruct** from HuggingFace.

## Features

- **Interactive Loop**: Continuously reads user input, sends it to the LLM, and prints responses
- **Device Detection**: Automatically uses CUDA (NVIDIA GPU), MPS (Apple Silicon), or CPU
- **Graph Visualization**: Saves a visual representation of the graph structure
- **Verbose Tracing Mode**: Built-in tracing system to monitor graph execution
- **Empty Input Handling**: Prevents empty inputs from reaching the LLM with a 3-way conditional routing branch

## Verbose/Quiet Modes

This implementation includes a **tracing system** that can be toggled at runtime:

- **`verbose`**: Type "verbose" to enable tracing output showing:
  - Current node being executed
  - Routing decisions made by conditional edges
  - Last 3 message exchanges with timestamps
  - Relevant metadata

- **`quiet`**: Type "quiet" to disable tracing output

You can also set the initial verbose mode when starting the agent by setting `verbose=True` or `verbose=False` in the initial state.

## How to Use

1. Run all cells in order
2. The agent will prompt you for input
3. Type your message and press Enter
4. Type "verbose" to enable tracing, "quiet" to disable it
5. Type "quit", "exit", or "q" to stop the agent
6. Empty inputs (just pressing Enter) will loop back to prompt without calling the LLM

In [13]:
%pip install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [14]:
%pip install torch transformers langchain-huggingface langgraph grandalf

In [15]:
# Import necessary libraries
import sys
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from datetime import datetime
from getpass import getpass

In [16]:
# Determine the best available device for inference
# Priority: CUDA (NVIDIA GPU) > MPS (Apple Silicon) > CPU
def get_device():
    """
    Detect and return the best available compute device.
    Returns 'cuda' for NVIDIA GPUs, 'mps' for Apple Silicon, or 'cpu' as fallback.
    """
    if torch.cuda.is_available():
        print("Using CUDA (NVIDIA GPU) for inference")
        return "cuda"
    elif torch.backends.mps.is_available():
        print("Using MPS (Apple Silicon) for inference")
        return "mps"
    else:
        print("Using CPU for inference")
        return "cpu"

In [17]:
# =============================================================================
# STATE DEFINITION
# =============================================================================
# The state is a TypedDict that flows through all nodes in the graph.
# Each node can read from and write to specific fields in the state.
# LangGraph automatically merges the returned dict from each node into the state.

class AgentState(TypedDict):
    """
    State object that flows through the LangGraph nodes.

    Original Fields:
    - user_input: The text entered by the user (set by get_user_input node)
    - should_exit: Boolean flag indicating if user wants to quit (set by get_user_input node)
    - llm_response: The response generated by the LLM (set by call_llm node)

    New Fields for Tracing:
    - verbose: Boolean flag to enable/disable tracing output
    - message_history: List of ALL conversation exchanges (no limit)
        Format: [{"user": "...", "llm": "..."}, ...]
        Note: All exchanges are stored and injected into LLM context.
              Only last 5 are displayed in trace output for readability.
    - current_node: Name of the currently executing node (for tracing)
    - route_taken: Last routing decision made (for tracing)

    Empty Input Detection:
    - is_empty_input: Boolean flag indicating if user provided empty input (set by get_user_input node)

    State Flow with 3-Way Conditional Branch:
        START → get_user_input → [3-way routing] → call_llm → print_response → get_user_input
                                      ↓    ↓    ↓           ↑_________________________________|
                                      ↓    ↓    ↓
                                      ↓    ↓    +--→ get_user_input (loop for empty/toggle)
                                      ↓    ↓
                                      ↓    +-------→ get_user_input (loop for empty/toggle)
                                      ↓
                                      +------------→ END (for quit/exit)
    """
    # Original fields
    user_input: str
    should_exit: bool
    llm_response: str

    # New fields for tracing
    verbose: bool
    message_history: list
    current_node: str
    route_taken: str

    # Empty input detection
    is_empty_input: bool

In [18]:
# =============================================================================
# TRACING UTILITY FUNCTIONS
# =============================================================================

def print_trace(state: AgentState, node_name: str):
    """
    Display tracing information if verbose mode is enabled.
    Shows current node, route taken, and last 5 message exchanges (for readability).
    """
    if not state.get("verbose", False):
        return
    
    print("\n" + "[TRACE] " + "=" * 50)
    print(f"[TRACE] NODE: {node_name}")
    print("[TRACE] " + "=" * 50)
    
    # Show route taken (if available)
    route = state.get("route_taken", "N/A")
    print(f"[TRACE] Route Taken: {route}")
    
    # Show message history (last 5 exchanges for readability)
    history = state.get("message_history", [])
    display_history = history[-5:]  # Cap display at 5, but ALL history is in LLM context
    
    total_exchanges = len(history)
    displayed_exchanges = len(display_history)
    
    if displayed_exchanges < total_exchanges:
        print(f"[TRACE] Message History (showing last {displayed_exchanges} of {total_exchanges} exchanges):")
    else:
        print(f"[TRACE] Message History ({total_exchanges} exchanges):")
    
    if not display_history:
        print("[TRACE]   (No messages yet)")
    else:
        start_idx = total_exchanges - displayed_exchanges + 1
        for idx, msg in enumerate(display_history, start_idx):
            user_msg = msg.get("user", "")
            llm_msg = msg.get("llm", "")
            
            # Truncate long messages for readability
            user_display = user_msg[:60] + "..." if len(user_msg) > 60 else user_msg
            llm_display = llm_msg[:60] + "..." if len(llm_msg) > 60 else llm_msg
            
            print(f"[TRACE]   {idx}.")
            print(f"[TRACE]      User: {user_display}")
            print(f"[TRACE]      LLM:  {llm_display}")
    
    print("[TRACE] " + "=" * 50 + "\n")


def update_message_history(state: AgentState, user_input: str, llm_response: str) -> list:
    """
    Update message history with a new exchange.
    Keeps ALL exchanges (no limit).
    
    Args:
        state: Current agent state
        user_input: User's input message
        llm_response: LLM's response message
    
    Returns:
        Updated message_history list
    """
    history = state.get("message_history", []).copy()
    
    # Create new message entry (simple format, no timestamp)
    new_message = {
        "user": user_input,
        "llm": llm_response
    }
    
    # Add new message to history
    history.append(new_message)
    
    # Return all exchanges (no limit)
    return history


def is_toggle_command(user_input: str) -> bool:
    """
    Check if the user input is a verbose/quiet toggle command.
    
    Args:
        user_input: The user's input string
    
    Returns:
        True if input is "verbose" or "quiet", False otherwise
    """
    return user_input.lower() in ["verbose", "quiet"]

In [19]:
# =============================================================================
# LLM CREATION
# =============================================================================

def create_llm():
    """
    Create and configure the LLM using HuggingFace's transformers library.
    Downloads llama-3.2-1B-Instruct from HuggingFace Hub and wraps it
    for use with LangChain via HuggingFacePipeline.
    """
    # Get the optimal device for inference
    device = get_device()

    # Model identifier on HuggingFace Hub
    model_id = "meta-llama/Llama-3.2-1B-Instruct"

    print(f"Loading model: {model_id}")
    print("This may take a moment on first run as the model is downloaded...")

    # Prompt for HuggingFace token (required for gated models like Llama)
    print("\nPlease enter your HuggingFace token:")
    print("(Get your token from: https://huggingface.co/settings/tokens)")
    hf_token = getpass("HF Token: ")

    # Load the tokenizer - converts text to tokens the model understands
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)

    # Load the model itself with appropriate settings for the device
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        token=hf_token,
        torch_dtype=torch.float16 if device != "cpu" else torch.float32,
        device_map=device if device == "cuda" else None,
    )

    # Move model to MPS device if using Apple Silicon
    if device == "mps":
        model = model.to(device)

    # Create a text generation pipeline that combines model and tokenizer
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=256,  # Maximum tokens to generate in response
        do_sample=True,      # Enable sampling for varied responses
        temperature=0.7,     # Controls randomness (lower = more deterministic)
        top_p=0.95,          # Nucleus sampling parameter
        pad_token_id=tokenizer.eos_token_id,  # Suppress pad_token_id warning
    )

    # Wrap the HuggingFace pipeline for use with LangChain
    llm = HuggingFacePipeline(pipeline=pipe)

    print("Model loaded successfully!")
    return llm

In [20]:
# =============================================================================
# GRAPH CREATION: NODE DEFINITIONS, ROUTING, AND ASSEMBLY
# =============================================================================

def create_graph(llm):
    """
    Create the LangGraph state graph with three nodes and conditional routing.
    
    Nodes:
    1. get_user_input: Reads input from stdin, handles toggle commands and empty input
    2. call_llm: Sends input to the LLM and gets response
    3. print_response: Prints the LLM's response to stdout
    
    Graph structure with 3-way conditional branch:
        START → get_user_input → [3-way routing] → call_llm → print_response → get_user_input
                                      ↓    ↓    ↓           ↑_________________________________|
                                      ↓    ↓    ↓
                                      ↓    ↓    +--→ get_user_input (loop for empty/toggle)
                                      ↓    ↓
                                      ↓    +-------→ get_user_input (loop for empty/toggle)
                                      ↓
                                      +------------→ END (for quit/exit)
    """
    
    # =========================================================================
    # NODE 1: get_user_input
    # =========================================================================
    def get_user_input(state: AgentState) -> dict:
        """
        Node that prompts the user for input via stdin.
        Handles quit commands, verbose/quiet toggle commands, and empty input.
        
        Reads state: verbose (for tracing)
        Updates state:
            - user_input: The raw text entered by the user
            - should_exit: True if user wants to quit
            - verbose: Updated if user types "verbose" or "quiet"
            - is_empty_input: True if user provided empty input
            - current_node: Set to "get_user_input"
        """
        # Print tracing info if verbose mode is enabled
        print_trace(state, "get_user_input")
        
        # Display banner before each prompt
        print("\n" + "=" * 50)
        print("Enter your text (or 'quit' to exit):")
        print("=" * 50)
        
        print("\n> ", end="")
        sys.stdout.flush()  # Flush before input to ensure all previous output is visible
        user_input = input()
        
        # Check if input is empty (after stripping whitespace)
        if not user_input.strip():
            print("\n[Empty input - please enter some text]")
            return {
                "user_input": "",
                "should_exit": False,
                "is_empty_input": True,
                "current_node": "get_user_input"
            }
        
        # Echo user input so it's visible in non-verbose mode
        print(f"\nYou: {user_input}")
        
        # Check if user wants to exit
        if user_input.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            return {
                "user_input": user_input,
                "should_exit": True,
                "is_empty_input": False,
                "current_node": "get_user_input"
            }
        
        # Check if user wants to toggle verbose mode (short circuit)
        if user_input.lower() == "verbose":
            print("[Tracing enabled]")
            return {
                "user_input": user_input,
                "should_exit": False,
                "verbose": True,
                "is_empty_input": False,
                "current_node": "get_user_input"
            }
        
        if user_input.lower() == "quiet":
            print("[Tracing disabled]")
            return {
                "user_input": user_input,
                "should_exit": False,
                "verbose": False,
                "is_empty_input": False,
                "current_node": "get_user_input"
            }
        
        # Normal input - proceed to LLM
        return {
            "user_input": user_input,
            "should_exit": False,
            "is_empty_input": False,
            "current_node": "get_user_input"
        }
    
    # =========================================================================
    # NODE 2: call_llm
    # =========================================================================
    def call_llm(state: AgentState) -> dict:
        """
        Node that invokes the LLM with the user's input and full conversation history.
        
        Reads state:
            - user_input: The text to send to the LLM
            - message_history: All previous conversation exchanges
            - verbose: For tracing
        Updates state:
            - llm_response: The text generated by the LLM
            - message_history: Updated with the new exchange
            - current_node: Set to "call_llm"
        """
        # Print tracing info if verbose mode is enabled
        print_trace(state, "call_llm")
        
        user_input = state["user_input"]
        
        # Build conversation history into prompt
        history = state.get("message_history", [])
        prompt_parts = []
        
        # Add system prompt to guide the model's behavior
        prompt_parts.append("You are a helpful AI assistant. Respond to the user's message with a single, direct response. Do not generate additional conversation turns or continue the dialogue beyond your immediate response.")
        prompt_parts.append("")  # Blank line after system prompt
        
        # Add all previous exchanges
        for exchange in history:
            prompt_parts.append(f"User: {exchange['user']}")
            prompt_parts.append(f"Assistant: {exchange['llm']}")
        
        # Add current user input
        prompt_parts.append(f"User: {user_input}")
        prompt_parts.append("Assistant:")
        
        # Join all parts with newlines
        prompt = "\n".join(prompt_parts)
        
        print("\nProcessing your input...")
        
        # Invoke the LLM with full conversation context (returns prompt + completion)
        full_response = llm.invoke(prompt)
        
        # Strip the prompt to get clean completion
        # Try exact prompt stripping first (most accurate)
        if full_response.startswith(prompt):
            response = full_response[len(prompt):].strip()
        elif "Assistant:" in full_response:
            # Fallback: split by last "Assistant:"
            response = full_response.split("Assistant:")[-1].strip()
        else:
            # Last resort: use full response
            response = full_response.strip()
        
        # Truncate at phantom conversation markers (LLM generating multi-turn conversations)
        # Stop at the first occurrence of "User:" or "Assistant:" in the completion
        for marker in ["\nUser:", "\nAssistant:", "\nA:"]:
            if marker in response:
                response = response.split(marker)[0].strip()
                break
        
        # Update message history with clean response
        updated_history = update_message_history(state, user_input, response)
        
        # Return updated fields
        return {
            "llm_response": response,
            "message_history": updated_history,
            "current_node": "call_llm"
        }
    
    # =========================================================================
    # NODE 3: print_response
    # =========================================================================
    def print_response(state: AgentState) -> dict:
        """
        Node that prints the LLM's response to stdout.
        
        Reads state:
            - llm_response: The text to print
            - verbose: For tracing
        Updates state:
            - current_node: Set to "print_response"
        """
        # Print tracing info if verbose mode is enabled
        print_trace(state, "print_response")
        
        print("\n" + "-" * 50)
        print("LLM Response:")
        print("-" * 50)
        print(state["llm_response"])
        print("-" * 50)
        
        # CRITICAL FIX: Flush output buffer to ensure response is visible
        sys.stdout.flush()
        
        # Add small delay to ensure output is rendered before next input
        time.sleep(0.1)
        
        # Add extra spacing before next node
        print()  # Extra newline for visual separation
        sys.stdout.flush()
        
        # Return updated node name
        return {"current_node": "print_response"}
    
    # =========================================================================
    # ROUTING FUNCTION
    # =========================================================================
    def route_after_input(state: AgentState) -> str:
        """
        Routing function with 3-way conditional branch:
        1. EXIT: User wants to quit
        2. LOOP_BACK: Empty input or toggle command
        3. PROCESS: Normal input to LLM
        
        Examines state:
            - should_exit: If True, terminate the graph
            - is_empty_input: If True, loop back to get_user_input
            - user_input: If "verbose" or "quiet", short circuit back to input
        
        Returns:
            - END: If user wants to quit
            - "get_user_input": If empty input or user toggled verbose/quiet
            - "call_llm": If user provided normal input
        """
        # Check if user wants to exit
        if state.get("should_exit", False):
            if state.get("verbose", False):
                print("[TRACE] Routing decision: EXIT")
            return END
        
        # Check if input is empty (NEW CHECK)
        if state.get("is_empty_input", False):
            if state.get("verbose", False):
                print("[TRACE] Routing decision: LOOP_BACK (empty input)")
            return "get_user_input"
        
        # Check if this was a toggle command (short circuit)
        if is_toggle_command(state.get("user_input", "")):
            if state.get("verbose", False):
                print("[TRACE] Routing decision: LOOP_BACK (toggle command)")
            return "get_user_input"
        
        # Default: Proceed to LLM
        if state.get("verbose", False):
            print("[TRACE] Routing decision: PROCESS to call_llm")
        return "call_llm"
    
    # =========================================================================
    # GRAPH CONSTRUCTION
    # =========================================================================
    # Create a StateGraph with our defined state structure
    graph_builder = StateGraph(AgentState)
    
    # Add all three nodes to the graph
    graph_builder.add_node("get_user_input", get_user_input)
    graph_builder.add_node("call_llm", call_llm)
    graph_builder.add_node("print_response", print_response)
    
    # Define edges:
    # 1. START -> get_user_input (always start by getting user input)
    graph_builder.add_edge(START, "get_user_input")
    
    # 2. get_user_input -> [conditional] -> call_llm OR get_user_input OR END
    #    Uses route_after_input to decide based on state
    graph_builder.add_conditional_edges(
        "get_user_input",      # Source node
        route_after_input,      # Routing function that examines state
        {
            "call_llm": "call_llm",              # Normal input -> proceed to LLM
            "get_user_input": "get_user_input",  # Toggle command or empty -> short circuit
            END: END                              # Quit command -> terminate graph
        }
    )
    
    # 3. call_llm -> print_response (always print after LLM responds)
    graph_builder.add_edge("call_llm", "print_response")
    
    # 4. print_response -> get_user_input (loop back for next input)
    graph_builder.add_edge("print_response", "get_user_input")
    
    # Compile the graph into an executable form
    graph = graph_builder.compile()
    
    return graph

In [21]:
# =============================================================================
# GRAPH VISUALIZATION
# =============================================================================

def save_graph_image(graph, filename="lg_graph.png"):
    """
    Generate a Mermaid diagram of the graph and save it as a PNG image.
    Uses the graph's built-in Mermaid export functionality.
    """
    try:
        # Get the Mermaid PNG representation of the graph
        # This requires the 'grandalf' package for rendering
        png_data = graph.get_graph(xray=True).draw_mermaid_png()
        
        # Write the PNG data to file
        with open(filename, "wb") as f:
            f.write(png_data)
        
        print(f"Graph image saved to {filename}")
    except Exception as e:
        print(f"Could not save graph image: {e}")
        print("You may need to install additional dependencies: pip install grandalf")

In [22]:
# =============================================================================
# MAIN EXECUTION FUNCTION
# =============================================================================

def main(verbose_mode=False):
    """
    Main function that orchestrates the simple agent workflow:
    1. Initialize the LLM
    2. Create the LangGraph
    3. Save the graph visualization
    4. Run the graph once (it loops internally until user quits)
    
    Args:
        verbose_mode: Initial verbose setting (True/False). Can be toggled at runtime.
    
    The graph handles all looping internally through its edge structure:
    - get_user_input: Prompts and reads from stdin, handles toggle commands and empty input
    - call_llm: Processes input through the LLM
    - print_response: Outputs the response, then loops back to get_user_input
    
    The graph only terminates when the user types 'quit', 'exit', or 'q'.
    """
    print("=" * 50)
    print("LangGraph Simple Agent with Llama-3.2-1B-Instruct")
    print("=" * 50)
    print()
    
    # Step 1: Create and configure the LLM
    llm = create_llm()
    
    # Step 2: Build the LangGraph with the LLM
    print("\nCreating LangGraph...")
    graph = create_graph(llm)
    print("Graph created successfully!")
    
    # Step 3: Save a visual representation of the graph before execution
    print("\nSaving graph visualization...")
    save_graph_image(graph)
    
    # Step 4: Run the graph - it will loop internally until user quits
    # Create initial state with all required fields
    initial_state: AgentState = {
        # Original fields
        "user_input": "",
        "should_exit": False,
        "llm_response": "",
        
        # New fields for tracing
        "verbose": verbose_mode,          # Set initial verbose mode
        "message_history": [],             # Empty history at start
        "current_node": "",                # Will be set by first node
        "route_taken": "START",            # Initial route
        
        # Empty input detection
        "is_empty_input": False            # Initially not empty
    }
    
    print(f"\nStarting agent (verbose mode: {verbose_mode})...")
    print("Type 'verbose' to enable tracing, 'quiet' to disable it.")
    print("Type 'quit', 'exit', or 'q' to stop.\n")
    
    # Single invocation - the graph loops internally via edges
    # The graph only exits when route_after_input returns END (user typed quit/exit/q)
    graph.invoke(initial_state)

In [25]:
# =============================================================================
# RUN THE AGENT
# =============================================================================

# Run the agent with verbose mode disabled (default)
# Change to verbose_mode=True to start with tracing enabled
# You can toggle verbose mode at runtime by typing "verbose" or "quiet"

if __name__ == "__main__":
    # Option 1: Start with tracing disabled (default)
    main(verbose_mode=False)
    
    # Option 2: Start with tracing enabled
    # main(verbose_mode=True)

LangGraph Simple Agent with Llama-3.2-1B-Instruct

Using CUDA (NVIDIA GPU) for inference
Loading model: meta-llama/Llama-3.2-1B-Instruct
This may take a moment on first run as the model is downloaded...

Please enter your HuggingFace token:
(Get your token from: https://huggingface.co/settings/tokens)


Device set to use cuda


Model loaded successfully!

Creating LangGraph...
Graph created successfully!

Saving graph visualization...
Graph image saved to lg_graph.png

Starting agent (verbose mode: False)...
Type 'verbose' to enable tracing, 'quiet' to disable it.
Type 'quit', 'exit', or 'q' to stop.


Enter your text (or 'quit' to exit):

> 
[Empty input - please enter some text]

Enter your text (or 'quit' to exit):

> 
[Empty input - please enter some text]

Enter your text (or 'quit' to exit):

> 
You: Here are some words

Processing your input...

--------------------------------------------------
LLM Response:
--------------------------------------------------
The words you've provided are: "Here are some words". I'm ready to assist. What's next?
--------------------------------------------------


Enter your text (or 'quit' to exit):

> 
You: quit
Goodbye!
